In [1]:
import numpy as np
import pandas as pd
import pickle

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

I0000 00:00:1790703203.071834   10918 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790703205.465954   10918 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790703212.664521   10918 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
df = pd.read_csv("Dataset/hour.csv")
print("Original Dataset Shape:", df.shape)

Original Dataset Shape: (17379, 17)


In [3]:
df["datetime"] = pd.to_datetime(df["dteday"]) + pd.to_timedelta(df["hr"], unit="h")

In [4]:
df = df.sort_values("datetime")
df = df.reset_index(drop=True)

In [5]:
df = df[["datetime", "cnt"]]
df = df.set_index("datetime")
data = df["cnt"].values.reshape(-1, 1)
print("Processed Data Shape:", data.shape)

Processed Data Shape: (17379, 1)


In [6]:
train_size = int(len(data) * 0.80)
train_data = data[:train_size]
test_data = data[train_size:]
print("Training Samples:", len(train_data))
print("Testing Samples:", len(test_data))

Training Samples: 13903
Testing Samples: 3476


In [7]:
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

In [8]:
SEQUENCE_LENGTH = 24
def create_sequences(data, sequence_length):
    X = []
    y = []
    for i in range(sequence_length, len(data)):
        X.append(data[i - sequence_length : i])
        y.append(data[i])
    return (np.array(X), np.array(y))

X_train, y_train = create_sequences(train_scaled, SEQUENCE_LENGTH)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

combined_data = np.concatenate((train_scaled[-SEQUENCE_LENGTH:], test_scaled), axis=0)
X_test, y_test = create_sequences(combined_data, SEQUENCE_LENGTH)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (13879, 24, 1)
y_train: (13879, 1)
X_test: (3476, 24, 1)
y_test: (3476, 1)


In [9]:
model = Sequential(
    [
        LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, 1)),
        LSTM(32),
        Dense(1),
    ]
)

E0000 00:00:1790703328.796277   10918 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/anas/LEARNING/GEN-AI/python/venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
model.compile(optimizer="adam", loss="mean_squared_error")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,345 (114.63 KB)

 Trainable params: 29,345 (114.63 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
early_stopping = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stopping],
    verbose=1,
)

Epoch 1/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 22s 39ms/step - loss: 0.0130 - val_loss: 0.0139
Epoch 2/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 27s 56ms/step - loss: 0.0056 - val_loss: 0.0094
Epoch 3/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 40s 53ms/step - loss: 0.0045 - val_loss: 0.0075
Epoch 4/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 33s 32ms/step - loss: 0.0040 - val_loss: 0.0068
Epoch 5/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 0.0032 - val_loss: 0.0047
Epoch 6/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 32ms/step - loss: 0.0028 - val_loss: 0.0046
Epoch 7/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 33ms/step - loss: 0.0027 - val_loss: 0.0046
Epoch 8/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 32ms/step - loss: 0.0026 - val_loss: 0.0044
Epoch 9/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 0.0025 - val_loss: 0.0047
Epoch 10/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 33ms/step - loss: 0.0025 - val_loss: 0.0041
Epoch 11/30
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 0.0023 - val_loss: 0.0042
Epoch 12/30
391/391 ━━━━━━━━━━

In [12]:
test_loss = model.evaluate(X_test, y_test, verbose=1)
print("Test Loss:", test_loss)
predictions_scaled = model.predict(X_test)
predictions = scaler.inverse_transform(predictions_scaled)
actual_values = scaler.inverse_transform(y_test)
mae = mean_absolute_error(actual_values, predictions)
mse = mean_squared_error(actual_values, predictions)
rmse = np.sqrt(mse)
print(f"MAE  : {mae:.2f} bikes/hour")
print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f} bikes/hour")

109/109 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0035
Test Loss: 0.003523432882502675
109/109 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step
MAE  : 37.73 bikes/hour
MSE  : 3220.19
RMSE : 56.75 bikes/hour


In [13]:
results = pd.DataFrame(
    {
        "Actual Bike Count": actual_values.flatten(),
        "Predicted Bike Count": predictions.flatten(),
    }
)
results["Absolute Error"] = abs(
    results["Actual Bike Count"] - results["Predicted Bike Count"]
)
print("Sample Predictions:")
print(results.head(20))

Sample Predictions:
    Actual Bike Count  Predicted Bike Count  Absolute Error
0               283.0            271.576965       11.423035
1               253.0            301.817108       48.817108
2               261.0            243.044418       17.955582
3               306.0            277.572693       28.427307
4               445.0            437.540070        7.459930
5               868.0            826.771606       41.228394
6               814.0            790.994629       23.005371
7               610.0            598.553406       11.446594
8               448.0            438.874695        9.125305
9               317.0            343.696045       26.696045
10              224.0            257.776184       33.776184
11              138.0            182.500656       44.500656
12               58.0            109.569862       51.569862
13               23.0             45.543472       22.543472
14                6.0             17.810228       11.810228
15                7.

In [14]:
model.save("model.keras")